In [ ]:
import re
import pandas as pd
import nltk
import pickle
import numpy as np
import torch
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score, f1_score
from scipy.stats import randint, uniform
from xgboost import XGBClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, hamming_loss, accuracy_score, jaccard_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
import tensorflow as tf
from sklearn.pipeline import Pipeline
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from sklearn.preprocessing import MultiLabelBinarizer
from gensim.models import Word2Vec, FastText
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Data Processing

In [2]:
filtered_lines = []
start_reading = False  

with open("genres.list", "r", encoding="ISO-8859-1") as f:
    for line in f:
        if "8: THE GENRES LIST" in line:
            start_reading = True 
            continue  
        
        if start_reading:
            filtered_lines.append(line.strip())


filtered_lines = filtered_lines[2:]
data = []
for i in filtered_lines:
    unpack = i.split("\t")
    data.append((unpack[0], unpack[-1])) 

genres_df = pd.DataFrame(data, columns=["Title", "Genre"])
genres_df['Title'] = genres_df['Title'].str.replace(r'\(.*\)', '', regex=True)
genres_df['Title'] = genres_df['Title'].str.replace(r'["{}]',"",regex=True).str.strip()
genres_df.head()

,Title,Genre
0,!Next?,Documentary
1,#1 Single,Reality-TV
2,#15SecondScare,Horror
3,#15SecondScare,Short
4,#15SecondScare,Thriller


In [3]:
with open("plot.list", "r", encoding="ISO-8859-1") as file:
    lines = file.readlines()

movies = []
current_movie = None
current_plot = []

for line in lines:
    mv_match = re.match(r"MV: (.*)", line)
    pl_match = re.match(r"PL: (.*)", line)

    if mv_match:
        if current_movie and current_plot:
            movies.append({"Title": current_movie, "Plot": " ".join(current_plot)})
        current_movie = mv_match.group(1).strip()
        current_plot = []

    elif pl_match and current_movie:
        current_plot.append(pl_match.group(1).strip())

if current_movie and current_plot:
    movies.append({"Title": current_movie, "Plot": " ".join(current_plot)})

plot_df = pd.DataFrame(movies)
plot_df['Title'] = plot_df['Title'].str.replace(r'\(.*\)', '', regex=True)
plot_df['Title'] = plot_df['Title'].str.replace(r'["{}]',"",regex=True).str.strip()
plot_df.head()

,Title,Plot
0,#7DaysLater,#7dayslater is an interactive comedy series fe...
1,#BlackLove,"This week, the five women work on getting what..."
2,#BlackLove,"With just one week left in the workshops, the ..."
3,#BlackLove,All of the women are struggling with what make...
4,#BlackLove,All of the women start making strides towards ...


In [4]:
df = genres_df.merge(plot_df, on='Title', how='inner')
df.head()

,Title,Genre,Plot
0,#7DaysLater,Comedy,#7dayslater is an interactive comedy series fe...
1,#BlackLove,Reality-TV,"This week, the five women work on getting what..."
2,#BlackLove,Reality-TV,"With just one week left in the workshops, the ..."
3,#BlackLove,Reality-TV,All of the women are struggling with what make...
4,#BlackLove,Reality-TV,All of the women start making strides towards ...


In [5]:
genres = ['Comedy', 'Romance', 'Action', 'Drama', 'Horror',
       'Family', 'Sci-Fi', 'Crime', 'Mystery', 'Biography',
       'Adventure', 'History', 'War', 'Fantasy',
       'Thriller', 'Sport', 'Documentary', 'Animation',
       'Adult', 'Western']

df = df[df["Genre"].isin(genres)]
df = df[df['Title'] != ""]
df = df.groupby(['Title', 'Plot'])['Genre'].agg(tuple).reset_index()
df["Genre"] = df["Genre"].apply(lambda x: tuple(set(x)))
df = df.drop_duplicates(subset=["Title", "Genre"])
df.head()

,Title,Plot,Genre
0,#,The night falls on the big city and a hooded f...,"(Comedy, Animation)"
1,#1,After reaching #1 at Mutual of New York and se...,"(Comedy, Documentary, Drama, Animation)"
2,#1 Cheerleader Camp,When they're hired to work at a cheerleading c...,"(Comedy,)"
3,#1 Serial Killer,Years of seething rage against the racism he's...,"(Horror,)"
4,#1 at the Apocalypse Box Office,"Jules is, self declared, the most useless pers...","(Comedy, Sci-Fi)"


In [6]:
df["num_cat"] = df["Genre"].apply(lambda x: len(x))
df = df[df["num_cat"] <= 6]
df = df.reset_index()
df.head()

,index,Title,Plot,Genre,num_cat
0,0,#,The night falls on the big city and a hooded f...,"(Comedy, Animation)",2
1,1,#1,After reaching #1 at Mutual of New York and se...,"(Comedy, Documentary, Drama, Animation)",4
2,2,#1 Cheerleader Camp,When they're hired to work at a cheerleading c...,"(Comedy,)",1
3,3,#1 Serial Killer,Years of seething rage against the racism he's...,"(Horror,)",1
4,4,#1 at the Apocalypse Box Office,"Jules is, self declared, the most useless pers...","(Comedy, Sci-Fi)",2


In [7]:
df_percentages = df["num_cat"].value_counts(normalize=True).reset_index()
df_percentages.columns = ["num_cat", "percentage"]
df_percentages["percentage"] = df_percentages["percentage"] * 100 
print(df_percentages)

   num_cat  percentage
0        1   49.195576
1        2   23.644912
2        3   15.550312
3        4    7.021097
4        5    3.063431
5        6    1.524672


In [8]:
df = df.loc[:, ["Plot", "Genre"]]
df.head()

,Plot,Genre
0,The night falls on the big city and a hooded f...,"(Comedy, Animation)"
1,After reaching #1 at Mutual of New York and se...,"(Comedy, Documentary, Drama, Animation)"
2,When they're hired to work at a cheerleading c...,"(Comedy,)"
3,Years of seething rage against the racism he's...,"(Horror,)"
4,"Jules is, self declared, the most useless pers...","(Comedy, Sci-Fi)"


# Tokenisation

In [9]:
nltk.data.path.append('/root/nltk_data')

nltk.download("punkt_tab")
nltk.download('stopwords')

[nltk_data] Error loading punkt_tab: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1000)>
[nltk_data] Error loading stopwords: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1000)>


False

In [10]:
stop_words = set(stopwords.words('english'))

# Function to Tokenize and Remove Stopwords
def clean_plot(text):
    tokens = word_tokenize(text.lower())
    tokens = [word for word in tokens if word.isalpha()]
    tokens = [word for word in tokens if word not in stop_words]
    return tokens if len(tokens) <= 200 else None

df["Tokenized"] = df["Plot"].apply(clean_plot)
df = df.dropna(subset=["Tokenized"])
df = df.reset_index()
df.to_csv("processed_plots.csv", index=False)

# Save as Pickle (for faster future use)
with open("processed_plots.pkl", "wb") as f:
    pickle.dump(df, f)

df.head()

,index,Plot,Genre,Tokenized
0,0,The night falls on the big city and a hooded f...,"(Comedy, Animation)","[night, falls, big, city, hooded, figure, emer..."
1,1,After reaching #1 at Mutual of New York and se...,"(Comedy, Documentary, Drama, Animation)","[reaching, mutual, new, york, several, financi..."
2,2,When they're hired to work at a cheerleading c...,"(Comedy,)","[hired, work, cheerleading, camp, summer, two,..."
3,3,Years of seething rage against the racism he's...,"(Horror,)","[years, seething, rage, racism, experienced, c..."
4,4,"Jules is, self declared, the most useless pers...","(Comedy, Sci-Fi)","[jules, self, declared, useless, person, post,..."


In [11]:
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df["Genre"])
genre_names = mlb.classes_

# TF-IDF

In [21]:
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X = vectorizer.fit_transform(df['Plot'])

In [22]:
X.shape

(277741, 5000)

In [23]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### XGBoost

In [ ]:
model = XGBClassifier(eval_metric='logloss')

# Train the model
model.fit(X_train, y_train)

# Predict on the test set
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
y_pred = model.predict(X_test)
J_Accuracy = jaccard_score(y_test, y_pred,average="samples")
f1 = f1_score(y_test, y_pred, average='micro')
print(f"Accuracy: {accuracy * 100:.2f}")
print(f"F1 Score: {f1:.4f}")
print(f"Jaccard Accuracy: {J_Accuracy * 100:.2f}%")

Accuracy: 22.51
Jaccard Accuracy: 36.11%


### SVC

In [27]:
model = Pipeline([
    ('clf', OneVsRestClassifier(LinearSVC()))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='micro')
J_Accuracy = jaccard_score(y_test, y_pred,average="samples")

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Jaccard Accuracy: {J_Accuracy * 100:.2f}%")

Accuracy: 0.2720
F1 Score: 0.4952
Jaccard Accuracy: 43.34%


### Logistic Regression

In [28]:
model = Pipeline([
    ('clf', OneVsRestClassifier(LogisticRegression(solver='liblinear')))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='micro')
J_Accuracy = jaccard_score(y_test, y_pred,average="samples")

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Jaccard Accuracy: {J_Accuracy * 100:.2f}%")

Accuracy: 0.2694
F1 Score: 0.5035
Jaccard Accuracy: 43.71%


## Word2vec

In [50]:
word2vec_model = Word2Vec(sentences=df["Tokenized"], vector_size=100, window=5, min_count=1, workers=4)

In [69]:
def get_sentence_embedding(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(100)

df["W2V Embedding"] = df["Tokenized"].apply(lambda x: get_sentence_embedding(x, word2vec_model))
X = np.vstack(df["W2V Embedding"].values)

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

Training samples: 255537
Testing samples: 28393


### XGBoost

In [ ]:
pipeline_xgb = Pipeline([
    ('pca', PCA()),
    ('xgb', XGBClassifier(objective="binary:logistic", use_label_encoder=False, eval_metric="logloss"))
])

param_dist_xgb = {
    'pca__n_components': randint(10, 80),
    'xgb__n_estimators': randint(50, 300),
    'xgb__max_depth': randint(2, 10),
    'xgb__learning_rate': uniform(0.01, 0.2),
}

random_search_xgb = RandomizedSearchCV(pipeline_xgb, param_distributions=param_dist_xgb, n_iter=5, 
                                       cv=3, scoring='accuracy', verbose=0, n_jobs=-1, random_state=42)

random_search_xgb.fit(X_train, y_train)

# Best parameters
print("Best XGBoost Parameters:", random_search_xgb.best_params_)

# Train with best params
xgb_best = random_search_xgb.best_estimator_
y_pred_xgb = xgb_best.predict(X_test)

/Users/chakir/Desktop/nlp/project/my_env/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [15:46:20] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/chakir/Desktop/nlp/project/my_env/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [15:46:21] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/chakir/Desktop/nlp/project/my_env/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [15:46:21] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/chakir/Desktop/nlp/project/my_env/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [15:46:21] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_lab

Best XGBoost Parameters: {'pca__n_components': 39, 'xgb__learning_rate': 0.052467822135655234, 'xgb__max_depth': 9, 'xgb__n_estimators': 237}
XGBoost Classification Report:
               precision    recall  f1-score   support

           0       0.67      0.24      0.36      2744
           1       0.55      0.04      0.07       447
           2       0.60      0.10      0.17      1966
           3       0.71      0.18      0.28      1819
           4       0.41      0.02      0.04      1383
           5       0.68      0.49      0.57      7944
           6       0.57      0.18      0.28      2006
           7       0.82      0.75      0.78      7191
           8       0.71      0.67      0.69     12030
           9       0.64      0.12      0.20      2211
          10       0.58      0.09      0.16      1580
          11       0.54      0.04      0.07      1311
          12       0.72      0.29      0.41      2054
          13       0.54      0.04      0.08      1407
          14   

/Users/chakir/Desktop/nlp/project/my_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [30]:
def metrics(y_test, y_pred):
    jaccard_weighted = jaccard_score(y_test, y_pred, average='samples')
    print(f"Jaccard Index (Samples): {jaccard_weighted:.4f}")

    hit_rate = np.mean(np.sum(y_test * y_pred, axis=1) > 0)
    print(f"Hit Rate: {hit_rate:.4f}")

print("XGBoost Classification Report:\n", classification_report(y_test, y_pred_xgb, target_names=genre_names))
metrics(y_test, y_pred_xgb)

XGBoost Classification Report:
               precision    recall  f1-score   support

      Action       0.67      0.24      0.36      2744
       Adult       0.55      0.04      0.07       447
   Adventure       0.60      0.10      0.17      1966
   Animation       0.71      0.18      0.28      1819
   Biography       0.41      0.02      0.04      1383
      Comedy       0.68      0.49      0.57      7944
       Crime       0.57      0.18      0.28      2006
 Documentary       0.82      0.75      0.78      7191
       Drama       0.71      0.67      0.69     12030
      Family       0.64      0.12      0.20      2211
     Fantasy       0.58      0.09      0.16      1580
     History       0.54      0.04      0.07      1311
      Horror       0.72      0.29      0.41      2054
     Mystery       0.54      0.04      0.08      1407
     Romance       0.52      0.12      0.20      3013
      Sci-Fi       0.72      0.25      0.37      1441
       Sport       0.60      0.27      0.37      

/Users/chakir/Desktop/nlp/project/my_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Random Forest

In [32]:
# Get the best PCA-reduced data
best_pca_n_components = random_search_xgb.best_params_['pca__n_components']
pca = PCA(n_components=best_pca_n_components)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_pca, y_train)
y_pred_rf = rf_model.predict(X_test_pca)

In [33]:
print("Random Forest Classification Report:\n", classification_report(y_test, y_pred_rf, target_names=genre_names))
metrics(y_test, y_pred_rf)

Random Forest Classification Report:
               precision    recall  f1-score   support

      Action       0.78      0.09      0.16      2744
       Adult       0.50      0.00      0.00       447
   Adventure       0.78      0.01      0.03      1966
   Animation       0.87      0.06      0.11      1819
   Biography       0.75      0.00      0.00      1383
      Comedy       0.71      0.36      0.48      7944
       Crime       0.67      0.04      0.08      2006
 Documentary       0.83      0.68      0.75      7191
       Drama       0.70      0.62      0.66     12030
      Family       0.79      0.03      0.05      2211
     Fantasy       0.72      0.01      0.03      1580
     History       0.75      0.00      0.00      1311
      Horror       0.85      0.06      0.11      2054
     Mystery       1.00      0.00      0.00      1407
     Romance       0.58      0.02      0.05      3013
      Sci-Fi       0.89      0.09      0.16      1441
       Sport       0.69      0.05      0.10

/Users/chakir/Desktop/nlp/project/my_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
X_train_gru = np.expand_dims(X_train_pca, axis=1)
X_test_gru = np.expand_dims(X_test_pca, axis=1)

gru_model = Sequential([
    GRU(100, input_shape=(1, best_pca_n_components), return_sequences=False),
    Dropout(0.3),
    Dense(100, activation='relu'),
    Dropout(0.3),
    Dense(y_train.shape[1], activation='sigmoid')
])


gru_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
gru_model.fit(X_train_gru, y_train, epochs=10, batch_size=32, validation_data=(X_test_gru, y_test))
y_pred_gru = (gru_model.predict(X_test_gru) > 0.5).astype(int)

Epoch 1/10


/Users/chakir/Desktop/nlp/project/my_env/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


7986/7986 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step - accuracy: 0.4377 - loss: 0.2361 - val_accuracy: 0.4794 - val_loss: 0.2021
Epoch 2/10
7986/7986 ━━━━━━━━━━━━━━━━━━━━ 7s 911us/step - accuracy: 0.4664 - loss: 0.2092 - val_accuracy: 0.4891 - val_loss: 0.2002
Epoch 3/10
7986/7986 ━━━━━━━━━━━━━━━━━━━━ 7s 930us/step - accuracy: 0.4696 - loss: 0.2074 - val_accuracy: 0.4903 - val_loss: 0.1997
Epoch 4/10
7986/7986 ━━━━━━━━━━━━━━━━━━━━ 7s 883us/step - accuracy: 0.4732 - loss: 0.2066 - val_accuracy: 0.4968 - val_loss: 0.1993
Epoch 5/10
7986/7986 ━━━━━━━━━━━━━━━━━━━━ 7s 866us/step - accuracy: 0.4732 - loss: 0.2055 - val_accuracy: 0.4947 - val_loss: 0.1989
Epoch 6/10
7986/7986 ━━━━━━━━━━━━━━━━━━━━ 7s 840us/step - accuracy: 0.4722 - loss: 0.2055 - val_accuracy: 0.4893 - val_loss: 0.1986
Epoch 7/10
7986/7986 ━━━━━━━━━━━━━━━━━━━━ 7s 891us/step - accuracy: 0.4753 - loss: 0.2050 - val_accuracy: 0.4949 - val_loss: 0.1985
Epoch 8/10
7986/7986 ━━━━━━━━━━━━━━━━━━━━ 7s 864us/step - accuracy: 0.4719 - loss: 0.2056

/Users/chakir/Desktop/nlp/project/my_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/chakir/Desktop/nlp/project/my_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
print("GRU Classification Report:\n", classification_report(y_test, y_pred_gru, target_names=genre_names))
metrics(y_test, y_pred_gru)

Jaccard Index (Samples): 0.4554
Hit Rate: 0.6842


## FastText

In [70]:
fasttext_model = FastText(
    sentences=df["Tokenized"], 
    vector_size=100, 
    window=5, 
    min_count=1, 
    workers=4
)

def get_sentence_embedding(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(100) 


df["FT Embedding"] = df["Tokenized"].apply(lambda x: get_sentence_embedding(x, fasttext_model))
X = np.vstack(df["FT Embedding"].values)

In [58]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

Training samples: 249966
Testing samples: 27775


In [59]:
xgb_model = XGBClassifier(objective="binary:logistic", use_label_encoder=False, eval_metric="logloss", n_estimators=100)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
print("XGBoost Classification Report:\n", classification_report(y_test, y_pred_xgb, target_names=genre_names))

/Users/chakir/Desktop/nlp/project/my_env/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [17:35:13] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


XGBoost Classification Report:
               precision    recall  f1-score   support

      Action       0.61      0.24      0.35      2612
       Adult       0.52      0.06      0.10       452
   Adventure       0.49      0.09      0.16      1814
   Animation       0.66      0.20      0.31      1807
   Biography       0.41      0.04      0.07      1414
      Comedy       0.65      0.48      0.55      7781
       Crime       0.51      0.17      0.25      2016
 Documentary       0.80      0.72      0.76      6991
       Drama       0.67      0.64      0.66     11630
      Family       0.56      0.12      0.19      2240
     Fantasy       0.51      0.09      0.16      1558
     History       0.41      0.04      0.07      1291
      Horror       0.66      0.28      0.39      2009
     Mystery       0.43      0.06      0.10      1408
     Romance       0.50      0.13      0.20      3017
      Sci-Fi       0.66      0.27      0.38      1440
       Sport       0.60      0.27      0.37      

/Users/chakir/Desktop/nlp/project/my_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [80]:
metrics(y_test, y_pred_xgb)

Jaccard Index (Samples): 0.4250
Hit Rate: 0.6672


In [60]:
X_train_gru = np.expand_dims(X_train, axis=1)
X_test_gru = np.expand_dims(X_test, axis=1)

gru_model = Sequential([
    GRU(128, input_shape=(1, 100), return_sequences=False), 
    Dropout(0.3),
    Dense(100, activation='relu'),
    Dropout(0.3),
    Dense(y_train.shape[1], activation='sigmoid')
])

gru_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
gru_model.fit(X_train_gru, y_train, epochs=10, batch_size=32, validation_data=(X_test_gru, y_test))
y_pred_gru = (gru_model.predict(X_test_gru) > 0.5).astype(int)
print("GRU Classification Report:\n", classification_report(y_test, y_pred_gru, target_names=genre_names))

/Users/chakir/Desktop/nlp/project/my_env/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
7812/7812 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step - accuracy: 0.4196 - loss: 0.2339 - val_accuracy: 0.4758 - val_loss: 0.2065
Epoch 2/10
7812/7812 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step - accuracy: 0.4628 - loss: 0.2109 - val_accuracy: 0.4769 - val_loss: 0.2046
Epoch 3/10
7812/7812 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step - accuracy: 0.4676 - loss: 0.2091 - val_accuracy: 0.4744 - val_loss: 0.2032
Epoch 4/10
7812/7812 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step - accuracy: 0.4694 - loss: 0.2083 - val_accuracy: 0.4781 - val_loss: 0.2020
Epoch 5/10
7812/7812 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step - accuracy: 0.4715 - loss: 0.2074 - val_accuracy: 0.4836 - val_loss: 0.2019
Epoch 6/10
7812/7812 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step - accuracy: 0.4740 - loss: 0.2061 - val_accuracy: 0.4843 - val_loss: 0.2010
Epoch 7/10
7812/7812 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step - accuracy: 0.4754 - loss: 0.2053 - val_accuracy: 0.4829 - val_loss: 0.2015
Epoch 8/10
7812/7812 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step - accuracy: 0.4757 - loss: 0.2057 - 

/Users/chakir/Desktop/nlp/project/my_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/chakir/Desktop/nlp/project/my_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [83]:
metrics(y_test, y_pred_gru)
loss, accuracy_gru = gru_model.evaluate(X_test_gru, y_test)
print(f"Accuracy: {accuracy_gru * 100:.2f}%")

Jaccard Index (Samples): 0.4353
Hit Rate: 0.6560
868/868 ━━━━━━━━━━━━━━━━━━━━ 0s 396us/step - accuracy: 0.4901 - loss: 0.1990
Accuracy: 48.73%


-------------------